## 1. Environment Setup

In [1]:
from google.colab import drive, userdata
from pathlib import Path
from zipfile import ZipFile
import pandas as pd
import sys

drive.mount('/content/drive')

DATA_ROOT = Path("/content/UCSD_Anomaly_Dataset/UCSD_Anomaly_Dataset")
PED1_PATH = DATA_ROOT / "UCSDped1"
PED2_PATH = DATA_ROOT / "UCSDped2"
ZIP_PATH = Path("/content/drive/MyDrive/SurveillanceAnomalyDetection/UCSD_Anomaly_Dataset.zip")
EXTRACT_PATH = Path("/content/UCSD_Anomaly_Dataset")

if not (PED1_PATH.exists() and PED2_PATH.exists()):
  with ZipFile(ZIP_PATH, "r") as z:
    z.extractall(EXTRACT_PATH)

GH_TOKEN = userdata.get("GH_PAT")
REPO_DIR = Path("/content/drive/MyDrive/SurveillanceAnomalyDetection/repo")
REPO_URL = f"https://{GH_TOKEN}@github.com/Rishabh-G-Shetye/SurveillanceAnomalyDetection.git"
if not REPO_DIR.exists():
  !git clone {REPO_URL} "{REPO_DIR}"
%cd "{REPO_DIR}"

sys.path.insert(0, str(REPO_DIR / "src"))

df = pd.read_csv(REPO_DIR / "data" / "metadata.csv")
from data.ucsd_dataset import PreprocessConfig, UCSDClipDataset

ped1_config = PreprocessConfig(target_size=(128, 128), window_length=8, stride=4)
train_ds = UCSDClipDataset(df, DATA_ROOT, "Ped1", "Train", ped1_config)
test_ds = UCSDClipDataset(df, DATA_ROOT, "Ped1", "Test", ped1_config)
print("Train clips:", len(train_ds), "| Test clips:", len(test_ds))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/SurveillanceAnomalyDetection/repo
Train clips: 1666 | Test clips: 1762


## 2. Module Directories  

In [2]:
import os
for sub in ["models", "training", "evaluation"]:
  os.makedirs(REPO_DIR / "src" / sub, exist_ok=True)
  (REPO_DIR / "src" / sub / "__init__.py").touch()

## 3. Base Model Interface

In [3]:
%%writefile "{REPO_DIR}/src/models/base.py"
"""
Common interface for manageing all the anomaly detection models in the project.

Every model implements:
- forward(clip): (B, T, 1, H, W) -> model-specific output
- compute_loss(clip): self-supervised loss, no labels needed
- anomaly_score(clip): (B,) float score, HIGHER = more anomalous

Purpose is such that a single trainer and a sinle evaluation module works with any model following this contract, so new architectures plug in without touching the training and the evaluation code.
"""

from abc import ABC, abstractmethod
import torch
import torch.nn as nn

class BaseAnomalyModel(nn.Module, ABC):
  """Abstract base class all the anomaly detection models must implement"""

  @abstractmethod
  def forward(self, clip: torch.Tensor) -> torch.Tensor:
    raise NotImplementedError

  @abstractmethod
  def compute_loss(self, clip: torch.Tensor) -> torch.Tensor:
    raise NotImplementedError

  @abstractmethod
  def anomaly_score(self, clip: torch.Tensor) -> torch.Tensor:
    raise NotImplementedError


Overwriting /content/drive/MyDrive/SurveillanceAnomalyDetection/repo/src/models/base.py


## 4. Model 1 - Baseline ConvAE

In [4]:
%%writefile "{REPO_DIR}/src/models/conv_ae.py"
import torch
import torch.nn as nn
import torch.nn.functional as F

from .base import BaseAnomalyModel

class ConvAE(BaseAnomalyModel):
    """Per-frame ConvAE. use_skip toggles U-Net-style skip connections;
    score_agg toggles clip-level scoring (mean vs. max over frames).
    Both are ablatable so their individual effect on AUC can be isolated."""
    def __init__(self, in_channels: int = 1, latent_channels: int = 64,
                 score_agg: str = "max", use_skip: bool = True):
        super().__init__()
        assert score_agg in ("mean", "max")
        self.score_agg = score_agg
        self.use_skip = use_skip

        self.enc1 = nn.Sequential(nn.Conv2d(in_channels, 32, 4, 2, 1), nn.ReLU())
        self.enc2 = nn.Sequential(nn.Conv2d(32, 64, 4, 2, 1), nn.ReLU())
        self.enc3 = nn.Sequential(nn.Conv2d(64, latent_channels, 4, 2, 1), nn.ReLU())

        dec2_in = 64 * 2 if use_skip else 64
        dec1_in = 32 * 2 if use_skip else 32
        self.dec3 = nn.Sequential(nn.ConvTranspose2d(latent_channels, 64, 4, 2, 1), nn.ReLU())
        self.dec2 = nn.Sequential(nn.ConvTranspose2d(dec2_in, 32, 4, 2, 1), nn.ReLU())
        self.dec1 = nn.Sequential(nn.ConvTranspose2d(dec1_in, in_channels, 4, 2, 1), nn.Tanh())

    def _forward_frame(self, x: torch.Tensor) -> torch.Tensor:
        e1 = self.enc1(x)
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)
        d3 = self.dec3(e3)
        d2 = self.dec2(torch.cat([d3, e2], dim=1) if self.use_skip else d3)
        d1 = self.dec1(torch.cat([d2, e1], dim=1) if self.use_skip else d2)
        return d1

    def forward(self, clip: torch.Tensor) -> torch.Tensor:
        B, T, C, H, W = clip.shape
        recon = self._forward_frame(clip.view(B * T, C, H, W))
        return recon.view(B, T, C, H, W)

    def compute_loss(self, clip: torch.Tensor) -> torch.Tensor:
        recon = self.forward(clip)
        return F.mse_loss(recon, clip)

    @torch.no_grad()
    def anomaly_score(self, clip: torch.Tensor) -> torch.Tensor:
        recon = self.forward(clip)
        per_frame_error = ((recon - clip) ** 2).mean(dim=(2, 3, 4))
        return per_frame_error.max(dim=1).values if self.score_agg == "max" else per_frame_error.mean(dim=1)

Overwriting /content/drive/MyDrive/SurveillanceAnomalyDetection/repo/src/models/conv_ae.py


In [5]:
from models.conv_ae import ConvAE
model = ConvAE()

## 5. Shared Training Loop

In [6]:
%%writefile "{REPO_DIR}/src/training/trainer.py"
"""
Generic training loop shared by every Base Anomaly Model implementation.

"""
from typing import Optional
import torch
from torch.utils.data import DataLoader

def train_model(
    model: torch.nn.Module,
    train_loader: DataLoader,
    num_epochs: int = 30,
    lr: float = 1e-3,
    device: Optional[str] = None,
    patience: int = 5,
) -> list:
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)

    history = []
    best_loss, epochs_without_improvement = float("inf"), 0

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        for batch in train_loader:
            clip = batch["clip"].to(device)
            optimizer.zero_grad()
            loss = model.compute_loss(clip)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * clip.size(0)
        epoch_loss = running_loss / len(train_loader.dataset)
        history.append(epoch_loss)
        scheduler.step(epoch_loss)
        print(f"Epoch {epoch+1}/{num_epochs} - loss: {epoch_loss:.6f} - lr: {optimizer.param_groups[0]['lr']:.2e}")

        if epoch_loss < best_loss - 1e-6:
            best_loss, epochs_without_improvement = epoch_loss, 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                print(f"Early stopping: no improvement for {patience} epochs.")
                break
    return history

Overwriting /content/drive/MyDrive/SurveillanceAnomalyDetection/repo/src/training/trainer.py


## 6. Shared Evaluation Metrics

In [7]:
%%writefile "{REPO_DIR}/src/evaluation/metrics.py"
"""
Evaluation metrics for anomaly detection: AUC-ROC, EER, precision/recall/F1
"""

import numpy as np
from sklearn.metrics import roc_auc_score, roc_curve, precision_recall_fscore_support


def compute_eer(labels: np.ndarray, scores: np.ndarray) -> float:
    """Equal Error Rate: point where false positive rate == false negative rate."""
    fpr, tpr, _ = roc_curve(labels, scores)
    fnr = 1 - tpr
    eer_idx = np.nanargmin(np.abs(fnr - fpr))
    return float((fpr[eer_idx] + fnr[eer_idx]) / 2)


def evaluate_scores(labels: np.ndarray, scores: np.ndarray, threshold: float = None) -> dict:
    """labels: 0/1 ground truth (1=anomalous). scores: higher = more anomalous."""
    auc = roc_auc_score(labels, scores)
    eer = compute_eer(labels, scores)

    if threshold is None:
        threshold = float(np.median(scores))
    preds = (scores >= threshold).astype(int)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary", zero_division=0
    )

    return {
        "auc_roc": auc, "eer": eer,
        "precision": precision, "recall": recall, "f1": f1,
        "threshold_used": threshold,
    }

Overwriting /content/drive/MyDrive/SurveillanceAnomalyDetection/repo/src/evaluation/metrics.py


## 7. Train ConvAE and Evaluate

In [8]:
import itertools
import torch
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader
from models.conv_ae import ConvAE
from training.trainer import train_model
from evaluation.metrics import evaluate_scores

device = "cuda" if torch.cuda.is_available() else "cpu"
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=2)

ablation_results = {}
for use_skip, score_agg in itertools.product([False, True], ["mean", "max"]):
    name = f"skip={use_skip}, agg={score_agg}"
    m = ConvAE(use_skip=use_skip, score_agg=score_agg)
    train_model(m, train_loader, num_epochs=15, patience=3)
    m.eval()
    scores, labels = [], []
    with torch.no_grad():
        for batch in test_loader:
            clip = batch["clip"].to(device)
            scores.append(m.anomaly_score(clip).cpu().numpy())
            labels.append(batch["clip_label"].numpy())
    scores, labels = np.concatenate(scores), np.concatenate(labels)
    ablation_results[name] = evaluate_scores(labels, scores)
    print(name, "->", ablation_results[name])

pd.DataFrame(ablation_results).T[["auc_roc", "eer", "f1"]]

Epoch 1/15 - loss: 0.054841 - lr: 1.00e-03
Epoch 2/15 - loss: 0.014636 - lr: 1.00e-03
Epoch 3/15 - loss: 0.009193 - lr: 1.00e-03
Epoch 4/15 - loss: 0.006781 - lr: 1.00e-03
Epoch 5/15 - loss: 0.005565 - lr: 1.00e-03
Epoch 6/15 - loss: 0.004733 - lr: 1.00e-03
Epoch 7/15 - loss: 0.004209 - lr: 1.00e-03
Epoch 8/15 - loss: 0.003809 - lr: 1.00e-03
Epoch 9/15 - loss: 0.003402 - lr: 1.00e-03
Epoch 10/15 - loss: 0.003146 - lr: 1.00e-03
Epoch 11/15 - loss: 0.002969 - lr: 1.00e-03
Epoch 12/15 - loss: 0.002771 - lr: 1.00e-03
Epoch 13/15 - loss: 0.002651 - lr: 1.00e-03
Epoch 14/15 - loss: 0.002473 - lr: 1.00e-03
Epoch 15/15 - loss: 0.002394 - lr: 1.00e-03
skip=False, agg=mean -> {'auc_roc': np.float64(0.6307578892297447), 'eer': 0.438918326447755, 'precision': 0.20998864926220204, 'recall': 0.5763239875389408, 'f1': 0.30782029950083195, 'threshold_used': 0.0028117941692471504}
Epoch 1/15 - loss: 0.047715 - lr: 1.00e-03
Epoch 2/15 - loss: 0.013540 - lr: 1.00e-03
Epoch 3/15 - loss: 0.009169 - lr: 1.0

,auc_roc,eer,f1
"skip=False, agg=mean",0.630758,0.438918,0.307820
"skip=False, agg=max",0.629271,0.448796,0.309484
"skip=True, agg=mean",0.630423,0.429388,0.316140
"skip=True, agg=max",0.622635,0.439612,0.309484


In [ ]:
# %cd "{REPO_DIR}"
# !git add notebooks/04_ucsd_model_experiments.ipynb
# !git commit -m "feat: Implement and evaluate ConvAE baseline model"
# !git push